# W5 Day2 (05/02 周五): RAG 全链路

## 学习目标
- **理论 (1h)**: 切分策略; Re-ranking; HyDE; Query 分解
- **实践 (2h)**: 实现完整 Naive RAG: 切分→Embed→FAISS→Re-rank→生成
- **产出物**: Naive RAG pipeline notebook

## 核心问题 (面试高频)
1. RAG 的完整链路是什么？每个环节有哪些选择？
2. 文档切分策略有哪些？语义切分 vs 递归切分？
3. Re-ranking 为什么必要？Bi-encoder vs Cross-encoder？
4. HyDE 是什么？为什么能提升检索效果？
5. RAG 和 Fine-tuning 什么时候选哪个？
6. RAG 的评估指标有哪些？

---

## 目录
1. [RAG 全链路概述](#1)
2. [文档切分策略](#2)
3. [检索 + Re-ranking](#3)
4. [HyDE 与 Query 变换](#4)
5. [完整 RAG Pipeline 实现](#5)
6. [RAG 评估](#6)
7. [总结 & 思考题](#7)

In [ ]:
import re
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple

TERRACOTTA = '#c2553a'
SAGE = '#5a6b4a'
INK = '#2d2a26'

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False

try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except ImportError:
    HAS_ST = False

print(f"FAISS: {HAS_FAISS}, Sentence-Transformers: {HAS_ST}")
print("=" * 60)

---
## 1. RAG 全链路概述 <a id='1'></a>

```
用户 Query
    ↓
[1] Query 变换 (HyDE / 分解 / 扩展)
    ↓
[2] 向量检索 (Embedding → FAISS → Top-K)
    ↓
[3] Re-ranking (Cross-encoder 精排)
    ↓
[4] Prompt 构建 (System + Context + Query)
    ↓
[5] LLM 生成 (带引用)
    ↓
回答
```

### 离线索引链路

```
文档 → 切分 (Chunking) → Embedding → 向量数据库
```

### RAG vs Fine-tuning

| | RAG | Fine-tuning |
|---|---|---|
| **知识更新** | 实时 (换文档即可) | 需要重训 |
| **成本** | 低 (只需 Embedding) | 高 (GPU 训练) |
| **幻觉** | 较少 (有引用) | 可能更多 |
| **私有数据** | 天然支持 | 需要训练数据 |
| **推理延迟** | 较高 (检索+生成) | 较低 |
| **适用** | 知识密集型、频繁更新 | 风格/格式调整 |

---
## 2. 文档切分策略 <a id='2'></a>

### 2.1 常见策略

| 策略 | 方法 | 优势 | 劣势 |
|---|---|---|---|
| **固定长度** | 按字符/token 数切分 | 简单 | 可能切断语义 |
| **递归切分** | 按层级分隔符 (\n\n→\n→。→ ) | 保留结构 | 需要调参 |
| **语义切分** | 用 Embedding 相似度找断点 | 语义完整 | 计算量大 |
| **文档结构** | 按标题/段落/章节 | 保留层次 | 依赖文档格式 |

### 2.2 Chunk Size 的选择

| Size | 适用场景 |
|---|---|
| **128-256 tokens** | 精确检索 (Q&A) |
| **512 tokens** | 通用 RAG (默认选择) |
| **1024+ tokens** | 长文档理解 (摘要) |

### 2.3 Overlap

相邻 chunk 重叠 10-20%，防止语义在边界处断裂。

In [ ]:
# ============ 文档切分实现 ============

class TextChunker:
    """文档切分器"""
    
    def __init__(self, chunk_size=256, overlap=32):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def fixed_length(self, text: str) -> List[str]:
        """固定长度切分"""
        chunks = []
        for i in range(0, len(text), self.chunk_size - self.overlap):
            chunk = text[i:i + self.chunk_size]
            if chunk:
                chunks.append(chunk)
        return chunks
    
    def recursive(self, text: str) -> List[str]:
        """递归切分: 按层级分隔符"""
        separators = ['\n\n', '\n', '。', '！', '？', '；', '，', ' ']
        return self._recursive_split(text, separators)
    
    def _recursive_split(self, text, separators):
        if len(text) <= self.chunk_size:
            return [text] if text.strip() else []
        
        # 找到第一个能切分的分隔符
        sep = None
        for s in separators:
            if s in text:
                sep = s
                break
        
        if sep is None:
            return self.fixed_length(text)
        
        # 按分隔符切分
        parts = text.split(sep)
        chunks = []
        current = ''
        for part in parts:
            if len(current) + len(part) + len(sep) <= self.chunk_size:
                current += part + sep if current else part
            else:
                if current:
                    chunks.append(current.strip())
                current = part
        if current.strip():
            chunks.append(current.strip())
        
        return chunks


# 测试
sample_doc = """建筑安全审查规范

第一章 总则
本规范适用于新建、扩建和改建的建筑工程的安全审查。所有建筑元素必须符合国家标准GB50016-2014的要求。
审查人员应当具备相应的专业资质，并持有有效的执业证书。

第二章 墙体审查
承重墙的最小厚度应为200mm（混凝土）或240mm（砖砌）。墙体的耐火极限不应低于2.0小时。
防火墙应从基础到屋顶全部采用不燃材料，且不应开设门窗洞口。如确需开设，应设置甲级防火门窗。

第三章 门窗审查
疏散通道的门净宽度不应小于0.9m，高度不应小于2.1m。
位于防火墙上的窗户，其与相邻窗户的间距不应小于1.0m。窗户的耐火极限应与墙体匹配。

第四章 楼梯审查
疏散楼梯的净宽度不应小于1.1m。楼梯间应设置防烟设施。
楼梯的踏步高度不应大于0.175m，踏步宽度不应小于0.25m。"""

chunker = TextChunker(chunk_size=100, overlap=20)

print("固定长度切分:")
for i, chunk in enumerate(chunker.fixed_length(sample_doc)):
    print(f"  [{i}] ({len(chunk)}字) {chunk[:50]}...")

print("\n递归切分:")
for i, chunk in enumerate(chunker.recursive(sample_doc)):
    print(f"  [{i}] ({len(chunk)}字) {chunk[:50]}...")

---
## 3. 检索 + Re-ranking <a id='3'></a>

### 3.1 Bi-encoder vs Cross-encoder

| | Bi-encoder | Cross-encoder |
|---|---|---|
| **编码方式** | Query 和 Doc 分别编码 | Query+Doc 拼接后编码 |
| **速度** | 快 (可预计算 Doc) | 慢 (每对都要算) |
| **精度** | 较低 | 较高 |
| **用途** | 初筛 (Top-100) | 精排 (Top-10) |
| **代表** | BGE, E5, GTE | ms-marco-MiniLM, BGE-reranker |

### 3.2 为什么需要 Re-ranking？

```
Bi-encoder 检索 Top-100 → 可能有不相关的
Cross-encoder 精排 Top-10 → 高精度结果

两阶段: 粗筛 (快但不准) → 精排 (准但慢)
```

### 3.3 其他检索增强方法

| 方法 | 思路 |
|---|---|
| **HyDE** | 让 LLM 先生成假设答案，用假设答案去检索 |
| **Query 分解** | 将复杂 Query 拆成多个子 Query |
| **Query 扩展** | 添加同义词/相关词 |
| **多路召回** | 向量检索 + BM25 + 知识图谱 |

---
## 4. HyDE 与 Query 变换 <a id='4'></a>

### HyDE (Hypothetical Document Embedding)

```
Query: "建筑安全审查中墙体厚度要求是什么？"

Step 1: LLM 生成假设答案
  → "根据GB50016-2014，承重墙最小厚度为200mm..."

Step 2: 用假设答案的 Embedding 去检索
  → 比用原始 Query 检索更准 (语义更接近文档)

Step 3: 检索到的真实文档 → LLM 生成最终答案
```

### 为什么 HyDE 有效？

- Query 通常很短，信息量少
- 假设答案更长、更详细，与文档的语义更接近
- 相当于做了 **Query Expansion**

In [ ]:
# ============ 完整 Naive RAG Pipeline ============

class SimpleEmbedder:
    """简易 Embedding (模拟或使用 sentence-transformers)"""
    def __init__(self, dim=384):
        self.dim = dim
        if HAS_ST:
            self.model = SentenceTransformer('BAAI/bge-small-zh-v1.5')
            self.dim = self.model.get_sentence_embedding_dimension()
    
    def encode(self, texts: List[str]) -> np.ndarray:
        if HAS_ST:
            return self.model.encode(texts, normalize_embeddings=True)
        else:
            # 简单的 TF-IDF 模拟
            np.random.seed(42)
            vectors = []
            for text in texts:
                # 基于字符的简单 hash
                vec = np.zeros(self.dim)
                for i, c in enumerate(text[:100]):
                    vec[hash(c) % self.dim] += 1
                norm = np.linalg.norm(vec)
                if norm > 0:
                    vec /= norm
                vectors.append(vec)
            return np.array(vectors, dtype='float32')


class SimpleVectorStore:
    """简易向量数据库"""
    def __init__(self, dim):
        self.dim = dim
        self.vectors = []
        self.documents = []
        self.index = None
    
    def add(self, docs: List[str], vectors: np.ndarray):
        self.documents.extend(docs)
        self.vectors.append(vectors)
        # 重建索引
        all_vecs = np.vstack(self.vectors).astype('float32')
        if HAS_FAISS:
            self.index = faiss.IndexFlatIP(self.dim)
            self.index.add(all_vecs)
        else:
            self.index = all_vecs
    
    def search(self, query_vec: np.ndarray, top_k: int = 5) -> List[Tuple[str, float]]:
        if HAS_FAISS:
            query_vec = query_vec.reshape(1, -1).astype('float32')
            scores, indices = self.index.search(query_vec, top_k)
            return [(self.documents[i], scores[0][j]) for j, i in enumerate(indices[0])]
        else:
            all_vecs = self.index
            scores = (all_vecs @ query_vec.flatten()).flatten()
            top_idx = np.argsort(scores)[::-1][:top_k]
            return [(self.documents[i], scores[i]) for i in top_idx]


class SimpleReranker:
    """简易 Re-ranker (基于交叉相似度)"""
    def __init__(self, embedder):
        self.embedder = embedder
    
    def rerank(self, query: str, docs: List[Tuple[str, float]], top_k: int = 3) -> List[Tuple[str, float]]:
        # 用 query embedding 和每个 doc embedding 的相似度重新排序
        q_vec = self.embedder.encode([query])[0]
        d_vecs = self.embedder.encode([d[0] for d in docs])
        scores = (d_vecs @ q_vec).tolist()
        reranked = [(docs[i][0], scores[i]) for i in range(len(docs))]
        reranked.sort(key=lambda x: -x[1])
        return reranked[:top_k]


class NaiveRAG:
    """完整的 Naive RAG Pipeline"""
    
    def __init__(self):
        self.embedder = SimpleEmbedder()
        self.chunker = TextChunker(chunk_size=200, overlap=30)
        self.vector_store = SimpleVectorStore(self.embedder.dim)
        self.reranker = SimpleReranker(self.embedder)
    
    def index_document(self, doc: str):
        """索引文档: 切分 → Embedding → 存储"""
        chunks = self.chunker.recursive(doc)
        vectors = self.embedder.encode(chunks)
        self.vector_store.add(chunks, vectors)
        return len(chunks)
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """检索: Embedding → 向量搜索 → Re-ranking"""
        q_vec = self.embedder.encode([query])[0]
        # 初筛
        candidates = self.vector_store.search(q_vec, top_k=top_k * 3)
        # 精排
        results = self.reranker.rerank(query, candidates, top_k=top_k)
        return results
    
    def generate_prompt(self, query: str, contexts: List[str]) -> str:
        """构建 RAG Prompt"""
        context_str = "\n---\n".join(contexts)
        return f"""基于以下参考信息回答问题。如果参考信息中没有相关内容，请说"我无法从提供的资料中找到答案"。

参考信息:
{context_str}

问题: {query}

回答:"""


# 构建 RAG 系统
rag = NaiveRAG()

# 索引建筑安全规范文档
documents = [
    sample_doc,
    """结构安全标准
混凝土结构的保护层厚度应满足: 梁不小于25mm，柱不小于30mm，板不小于15mm。
钢筋的锚固长度不应小于30倍钢筋直径。焊接接头的搭接长度不应小于10倍钢筋直径。
预应力混凝土构件的张拉控制应力不应超过钢筋标准强度的0.75倍。""",
    """消防设施审查
每层建筑面积超过1500m²的建筑应设置自动喷水灭火系统。消防车道的宽度不应小于4m，转弯半径不应小于12m。
室内消火栓的间距不应超过30m。消防应急照明的持续时间不应少于90分钟。""",
]

total_chunks = 0
for doc in documents:
    n = rag.index_document(doc)
    total_chunks += n
    print(f"索引文档: {n} 个 chunks")
print(f"总计: {total_chunks} 个 chunks")

# 测试查询
queries = [
    "承重墙的最小厚度是多少？",
    "疏散楼梯有什么要求？",
    "消防车道的宽度要求？",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    results = rag.retrieve(query, top_k=2)
    print(f"检索结果 (Top 2):")
    for i, (doc, score) in enumerate(results, 1):
        print(f"  [{i}] score={score:.3f}: {doc[:80]}...")
    prompt = rag.generate_prompt(query, [r[0] for r in results])
    print(f"\n生成的 Prompt (前200字):")
    print(f"  {prompt[:200]}...")

---
## 6. RAG 评估 <a id='6'></a>

### 评估维度

| 维度 | 指标 | 说明 |
|---|---|---|
| **检索质量** | Recall@K, MRR, NDCG | 检索到的文档是否相关 |
| **生成质量** | Faithfulness, Relevance | 生成的答案是否忠实于上下文 |
| **端到端** | Answer Correctness | 最终答案是否正确 |

### RAGAS 框架

- **Faithfulness**: 答案是否基于检索到的上下文 (不幻觉)
- **Answer Relevance**: 答案是否和问题相关
- **Context Precision**: 检索到的上下文中有多少是相关的
- **Context Recall**: 相关的上下文有多少被检索到了

---
## 7. 总结 & 思考题 <a id='7'></a>

### 今日核心知识点

1. **RAG 链路**: 切分→Embed→检索→Re-rank→生成
2. **切分策略**: 递归切分最实用，Chunk Size 256-512 tokens
3. **Re-ranking**: Bi-encoder 粗筛 + Cross-encoder 精排
4. **HyDE**: LLM 生成假设答案再检索，提升语义匹配
5. **RAG vs Fine-tuning**: RAG 适合知识密集+频繁更新，FT 适合风格调整
6. **评估**: RAGAS (Faithfulness/Relevance/Precision/Recall)

### 面试高频问题

1. **RAG 的完整链路？** 切分→Embed→索引→检索→Re-rank→Prompt→生成
2. **Chunk Size 怎么选？** 256-512 tokens，精确 QA 用小 chunk，长文理解用大 chunk
3. **为什么需要 Re-ranking？** Bi-encoder 精度不够，Cross-encoder 精排提升 10-20%
4. **RAG 和 Fine-tuning 怎么选？** RAG: 知识更新快/成本低; FT: 风格调整/推理快
5. **RAG 的失败模式？** 检索不到/检索到但不相关/生成时忽略上下文

In [ ]:
print("=" * 60)
print("W5 Day2 完成!")
print("=" * 60)
print("""
今日成果:
  ✅ 文档切分器: 固定长度 + 递归切分
  ✅ 向量数据库: SimpleVectorStore (FAISS backend)
  ✅ Re-ranker: 基于 Embedding 相似度的精排
  ✅ 完整 Naive RAG Pipeline (切分→Embed→检索→Re-rank→生成)
  ✅ RAG Prompt 模板

关键结论:
  • 递归切分 > 固定长度切分 (保留语义完整性)
  • Re-ranking 是 RAG 质量的关键环节
  • RAG 的上限由检索质量决定
  • HyDE 可以显著提升短 Query 的检索效果
""")